In [1]:
pip install alpaca-py


   -------------------- ------------------- 1/2 [alpaca-py]
   ---------------------------------------- 2/2 [alpaca-py]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from alpaca.trading.client import TradingClient
from dotenv import load_dotenv
import os
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from datetime import timedelta, datetime

load_dotenv()
API_KEY = os.getenv("API_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")

client = TradingClient(API_KEY, SECRET_KEY, paper=True)
data_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

def get_live_bars(symbol, lookback=5):
    request = StockBarsRequest(
        symbol_or_symbols=symbol,
        timeframe=TimeFrame(5, TimeFrameUnit.Minute),
        start=datetime.now() - timedelta(days=lookback)
    )
    bars = data_client.get_stock_bars(request)
    return bars.df

account = client.get_account()
print(account)
df = get_live_bars("AAPL")
df  # This actually prints the dataframe in the correct format.  Better than both df.head() which prints the first 5 rows and print(df) oddly enough

id=UUID('c74df5a0-1b68-4c2c-a6af-dcf26673433d') account_number='PA3X8KRAB0TK' status=<AccountStatus.ACTIVE: 'ACTIVE'> crypto_status=<AccountStatus.ACTIVE: 'ACTIVE'> currency='USD' buying_power='398440.16' regt_buying_power='199220.08' daytrading_buying_power=None non_marginable_buying_power='99610.04' cash='99220.83' accrued_fees='0' pending_transfer_out=None pending_transfer_in=None portfolio_value='99999.25' pattern_day_trader=None trading_blocked=False transfers_blocked=False account_blocked=False created_at=datetime.datetime(2026, 6, 10, 16, 10, 57, 578916, tzinfo=TzInfo(0)) trade_suspended_by_user=False multiplier='4' shorting_enabled=True equity='99999.25' last_equity='100009.75' long_market_value='778.42' short_market_value='0' initial_margin='389.21' maintenance_margin='389.21' last_maintenance_margin='394.46' sma='100013.19' daytrade_count=None options_buying_power='99610.04' options_approved_level=3 options_trading_level=3


open    high       low     close   volume  \
symbol timestamp                                                                
AAPL   2026-07-13 08:00:00+00:00  315.26  316.14  314.7300  315.7200  61736.0   
       2026-07-13 08:05:00+00:00  315.79  316.08  314.8300  316.0800   7362.0   
       2026-07-13 08:10:00+00:00  316.18  316.20  315.8100  316.2000   4211.0   
       2026-07-13 08:15:00+00:00  316.20  316.20  315.8144  315.8144   2984.0   
       2026-07-13 08:20:00+00:00  315.92  315.97  315.8100  315.9700    741.0   
...                                  ...     ...       ...       ...      ...   
       2026-07-16 22:20:00+00:00  333.77  333.89  333.5671  333.6600   9107.0   
       2026-07-16 22:25:00+00:00  333.66  333.84  333.5408  333.8400   6162.0   
       2026-07-16 22:30:00+00:00  333.85  333.85  333.7000  333.7000   6179.0   
       2026-07-16 22:35:00+00:00  333.68  333.81  333.6100  333.6800   4918.0   
       2026-07-16 22:40:00+00:00  333.59  333.85  333.3500  333.7800  24492.0   

                                  trade_count        vwap  
symbol timestamp                                           
AAPL   2026-07-13 08:00:00+00:00       3461.0  315.483540  
       2026-07-13 08:05:00+00:00        305.0  315.821184  
       2026-07-13 08:10:00+00:00        122.0  316.074356  
       2026-07-13 08:15:00+00:00         85.0  316.091997  
       2026-07-13 08:20:00+00:00         37.0  315.917352  
...                                       ...         ...  
       2026-07-16 22:20:00+00:00        175.0  333.744701  
       2026-07-16 22:25:00+00:00        124.0  333.695659  
       2026-07-16 22:30:00+00:00        137.0  333.831993  
       2026-07-16 22:35:00+00:00        116.0  333.649252  
       2026-07-16 22:40:00+00:00        255.0  333.649138  

[748 rows x 7 columns]

In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

def calculate_macd(df, fast=12, slow=26, signal=9):
    # calculate fast and slow EMAs
    df['ema_fast'] = df['close'].ewm(span=fast, adjust=False).mean()
    df['ema_slow'] = df['close'].ewm(span=slow, adjust=False).mean()
    
    # calculate MACD line (fast minus slow)
    df['macd'] = df['ema_fast'] - df['ema_slow']
    
    # calculate signal line (EMA of MACD line)
    df['signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    
    # calculate histogram (MACD minus signal)
    df['hist'] = df['macd'] - df['signal']

    return df

def plot_macd(df):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    # plot price on top
    ax1.plot(df.index, df['close'], label='Close')
    ax1.set_title('Price')
    ax1.legend()

    # plot MACD line and signal line in middle
    ax2.plot(df.index, df['macd'], label='MACD')
    ax2.plot(df.index, df['signal'], label='Signal')

    # plot histogram on bottom
    ax2.bar(df.index, df['hist'], label='Histogram', alpha=0.3)
    ax2.set_title('MACD')
    ax2.legend()

    plt.tight_layout()
    plt.show()

    pass

# download data
# ticker = "AAPL"
# df = yf.download(ticker, start="2023-01-01", end="2024-01-01")

# calculate and plot
df = calculate_macd(get_live_bars("AAPL"))
# plot_macd(df)

In [ ]:
from datetime import datetime, time as dtime
from zoneinfo import ZoneInfo
from alpaca.common.exceptions import APIError
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

def run_signals(df, symbol, qty=1, actionable=False):
    latest_hist = df['hist'].iloc[-1]
    previous_hist = df['hist'].iloc[-2]

    if previous_hist > 0 and latest_hist <= 0:
        action = 'SELL'
    elif previous_hist < 0 and latest_hist >= 0:
        action = 'BUY'
    else:
        action = 'HOLD'
    
    try:
        position = client.get_open_position(symbol)
        qty_held = int(position.qty)
    except APIError:
        qty_held = 0

    if action == 'SELL' and qty_held > 0:
        final_action = 'SELL'
    elif action == 'BUY' and qty_held == 0:
        final_action = 'BUY'
    else:
        final_action = 'HOLD'

    if final_action in ['BUY', 'SELL']:
        if actionable:
            side = OrderSide.BUY if final_action == 'BUY' else OrderSide.SELL
            order_data = MarketOrderRequest(
                symbol=symbol,
                qty=qty,
                side=side,
                time_in_force=TimeInForce.DAY
            )
            client.submit_order(order_data=order_data)
            print(f"Submitted {final_action} order for {qty} share(s) of {symbol}")
        else:
            print(f"[DRY RUN] Would {final_action} {qty} share(s) of {symbol}")

    return final_action


def is_market_open():
    now = datetime.now(ZoneInfo("America/New_York"))
    open_time = dtime(9, 30)
    close_time = dtime(16, 0)
    if now.weekday() < 5 and (open_time <= now.time() <= close_time):
        # check to see if there are market actions to take since the market is open
        return True
    return False

symbols = ["AAPL", "MSFT", "GOOGL", "JPM", "XOM"]

def job():
    if not is_market_open():
        return
    for symbol in symbols:
        # next: call get_live_bars, run that through calculate_macd, and finish with run_signals
        print(f"Would process {symbol}")

The market is closed
